In [ ]:
import scipy
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

from sklearn import linear_model    # Herramientas de modelos lineales
from sklearn.metrics import mean_squared_error, r2_score    # Medidas de desempeño
from sklearn.preprocessing import PolynomialFeatures    # Herramientas de polinomios

from sklearn.model_selection import train_test_split, KFold, cross_val_score

from formulaic import Formula

In [ ]:
# Si necesitan instalar algún paquete
#!pip install gapminder
#!pip install formulaic

In [ ]:
# Si necesitan cambiar de directorio de trabajo
#import os
#print(pwd)
#os.chdir('./notebooks')

# Interacciones entre variables

In [ ]:
# Consideramos un ejemplo con variables sintéticas

df = pd.DataFrame({
    'y': [11, 10.5, 7.8, 9.6, 2.1, 4.3],
    'x1': ['Y', 'Y', 'N', 'N', 'Y', 'Y'],
    'x2': [0.3, 0.1, 0.2, 0.7, 0.6, 0.6],
    'x3': [100, 121, 17, 89, 45, 32],
    'x4': ['A', 'B', 'C', 'C', 'A', 'C'],
})
df

In [ ]:
# Queremos modelar y usando la interacción entre x2 y x3.
# Por ejemplo, x2 es el porcentaje de grasa de un alimento, x3 es el peso del alimento 
y, X = (
    Formula('y ~ x2 : x3')
    .get_model_matrix(df)
)
display(X.head()) 

In [ ]:
# Para no abusar de formulaic... cómo podemos agregar la variable grasa_total a mano?


In [ ]:
# Si queremos conservar también las variables originales...
y, X = (
    Formula('y ~ x2 * x3')
    .get_model_matrix(df)
)
display(X) 

In [ ]:
# Y si queremos el x3 y la interacción entre x2 y x3 pero no queremos x2?
y, X = (
    Formula('y ~ ???')
    .get_model_matrix(df)
)
display(X)

In [ ]:
# Queremos agregar ahora la interacción entre x1 y x2.
y, X = (
    Formula('y ~ x1 : x2')
    .get_model_matrix(df)
)
display(X)

# Explicar qué columnas nos creó formulaic. Porqué crea dos y no una sola?

In [ ]:
# Qué esperamos en este caso?
y, X = (
    Formula('y ~ x1 * x2')
    .get_model_matrix(df)
)
display(X)

In [ ]:
# Que va a pasar en este caso?
y, X = (
    Formula('y ~ x1 * x2 - x2')
    .get_model_matrix(df)
)
display(X)

In [ ]:
# Observamos que sacamos una variable pero formulaic nos agrego otra!

In [ ]:
# Agreguemos ahora la interacción entre x3 y x4
# Qué esperamos acá?

y, X = (
    Formula('y ~ x3 : x4')
    .get_model_matrix(df)
)
display(X)

# Interacciones entre variables y la paradoja de Simpson.

Queremos estudiar la relación entre la longitud y la profundidad del pico de los pingüinos.

Vamos a utilizar
x = bill_length_mm,  y = bill_depth_mm

In [ ]:
penguins = sns.load_dataset("penguins")
penguins.head()

In [ ]:
# Vemos que hay datos faltanes.
# Eliminamos las filas con datos faltantes y reseteamos indices (muy importante para graficar predicciones!)
penguins = penguins.dropna().reset_index()
penguins

In [ ]:
# Ajustamos un modelo lineal y calculamos el coeficiente de correlación R^2
y, X = (
    Formula('bill_depth_mm ~ bill_length_mm')
    .get_model_matrix(penguins)
)
display(X.head()) 

In [ ]:
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo.fit(X, y)   # Realizamos el ajuste
print("Coeficientes:", modelo.coef_)

y_pred = modelo.predict(X)
# Calculando el R^2
r2 = r2_score(y, y_pred)
print('R^2: ', r2)

# Calculando el ECM
ecm = mean_squared_error(y, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

Si analizamos rápidamente estos resultados diríamos que no hay relación entre el largo y la profundidad... (o que hay correlación negativa porque la pendiente es negativa). Resulta un poco extraño...

¿Cómo podemos analizar mejor qué está pasando?

.

.

.

.

.

.

.

.

.

.

.

.

.

.

.




Realicemos un gráfico!

In [ ]:
(
    so.Plot(data = penguins, x = "bill_length_mm", y = "bill_depth_mm")
    .add(so.Dot())
    .add(so.Line(), so.PolyFit(1))
)

El gráfico confirma la correlación negativa, pero notan algo raro? Tal vez hay algo que no estamos teniendo en cuenta?

.

.

.

.

.


.

.

.

.

.


.

.

.

.

.



Repetimos el gráfico coloreando los puntos según la especie.

In [ ]:
(
    so.Plot(data = penguins, x = "bill_length_mm", y = "bill_depth_mm")
    .add(so.Dot(), color = "species")
    .add(so.Line(), so.PolyFit(1))
)

En este gráfico por especie vemos dentro de cada especie puede haber correlación. Verificamos agregando los ajustes por especie.

In [ ]:
(
    so.Plot(data = penguins, x = "bill_length_mm", y = "bill_depth_mm", color = "species")
    .add(so.Dot())
    .add(so.Line(), so.PolyFit(1))
)

Ahora las rectas tienen pendiente positiva! Al considerar todas las especies al mismo tiempo, no podíamos ver esta correlación.

## La paradoja de Simpson
La paradoja de Simpson es un fenómeno estadístico en el cual una relación entre variables aparece, desaparece o se revierte al dividir a la población en subpoblaciones.

**Ejemplo.** Veamos otro ejemplo simulado.
Generamos dos poblaciones distribuidas aleatoriamente alrededor de dos centros.

In [ ]:
from sklearn.datasets import make_blobs
centers = [[2, 2], [6, 6]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=0.4, random_state=0)
x = X[:,0]
y = X[:,1]

In [ ]:
(
    so.Plot(x = x, y = y)
    .add(so.Dot(), color = labels_true)
    .add(so.Line(), so.PolyFit(1))
)

En este ejemplo, podríamos decir que hay correlación entre la variable $x$ y la variable $y$?

Calculemos el R^2...

In [ ]:
modelo = linear_model.LinearRegression()    # Inicializamos un modelo de Regresion Lineal
modelo.fit(pd.DataFrame(x), y)   # Realiza

print("Coeficientes:", modelo.coef_)

# Medidas de bondad

y_pred = modelo.predict(pd.DataFrame(X[:,0]))

# Calculando el R^2
r2 = r2_score(X[:,1], y_pred)
print('R^2: ', r2)

Los datos parecen altamente correlacionados. Pero si separamos por grupo...

In [ ]:
(
    so.Plot(x = x, y = y)
    .add(so.Dot(), color = labels_true)
    .add(so.Line(), so.PolyFit(1), group = labels_true)
)

Vemos en el gráfico que la pendiente ahora pasa a ser casi 0.

Enseguida veremos como calcular estas pendientes usando modelos con interacciones.

Antes...

## Ejercicio

Utilizamos el dataset "SAT.csv".

Este conjunto de datos contiene métricas a nivel estatal para los 50 estados de EE. UU. 

| Variable | Descripción |
| :--- | :--- |
| **`state`** | Nombre del estado de EE. UU. |
| **`expend`** | Gasto corriente por alumno en asistencia diaria promedio (en miles de USD). |
| **`ratio`** | Promedio de alumnos por profesor en escuelas secundarias públicas. |
| **`salary`** | Salario anual promedio estimado de los profesores (en miles de USD). |
| **`frac`** | Porcentaje de graduados elegibles que tomaron el examen SAT. |
| **`verbal`** | Puntaje promedio estatal en la sección Verbal del SAT. |
| **`math`** | Puntaje promedio estatal en la sección de Matemáticas del SAT. |
| **`sat`** | Puntaje total promedio del SAT (Verbal + Matemáticas). |

Queremos ver la relación entre el salario promedio de los docentes en cada estado y el puntaje promedio en el SAT.

Si analizamos solo esas dos variables, la correlación es positiva o negativa?

¿Habrá alguna variable "confusora" que nos esté llevando a una conclusión incorrecta? ¿Cómo lo podemos explicar?


In [ ]:
#url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/mosaicData/SAT.csv"
#sat = pd.read_csv(url, index_col=0)
sat = pd.read_csv("sat.csv").set_index("rownames")
sat

Ahora sí vamos a las fórmulas...

## Modelos de regresión lineal con interacciones 
¿Cómo podemos construir nosotros estos modelos y calcular los coeficientes y el R^2?

Recordemos las operaciones que nos permite hacer Formulaic.

| Operador | Ejemplo          | Función                                                                                           |
|:---------|:-----------------|:---------------------------------------------------------------------------------------------------|
| ~        | y ~ x            | Separa la variable (y) respuesta a la izquierda, de el/los predictor/es a la derecha (x).       |
| +        | y ~ x + z        | Adiciona (suma) términos al modelo.                                                              |
| :        | y ~ x : z        | Interacción entre términos. y es lineal en x ⋅ z.                                                |
| *        | y ~ x * z        | Combina adición e interacción entre términos. y ~ x * z es equivalente a y ~ x + z + x : z       |

In [ ]:
# Comenzamos con un modelo simple:
# agregamos la especie como una variable más del modelo.
y, X = Formula("bill_depth_mm ~ bill_length_mm + species").get_model_matrix(penguins)
X.head()

In [ ]:
# Ajustamos el modelo 
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo.fit(X, y)   # Realiza
print("Coeficientes:", modelo.coef_)

In [ ]:
# Graficamos este modelo

# Notemos que queremos graficar un modelo con DOS VARIABLES EXPLICATIVAS. 
# Si fueran dos variables explicativas numéricas necesitamos hacer un gráfico en 3 dimensiones y = f(x_1, x_2) (o z = f(x,y))
# Pero en nuestro caso, una variable es categórica, cómo podemos hacerlo en este caso?

# Para eso, agregamos al dataframe de pingüinos nuestras predicciones
# Agregamos la variable y_pred a los datos para poder graficar con seaborn
datos = penguins.copy()

y_pred = modelo.predict(X)
datos["predicciones"] = y_pred
datos.head()

In [ ]:
# Realizamos el gráfico agregando una marca de línea para el ajuste del modelo.
(
    so.Plot(data = datos, x = "bill_length_mm")
    .add(so.Dot(), y = "bill_depth_mm", color = "species")
    .add(so.Line(color = "r"), y = "predicciones") 
)


In [ ]:
# No es lo que queriamos... El gráfico de lineas va saltando de una especie a otra.

# Para entender este gráfico, usemos puntos en vez de líneas

# Realizamos el gráfico agregando una marca de línea para el ajuste del modelo.
(
    so.Plot(data = datos, x = "bill_length_mm")
    .add(so.Dot(), y = "bill_depth_mm", color = "species")
    .add(so.Dot(color = "r"), y = "predicciones") 
)


In [ ]:
# Cómo podemos cambiar los puntos por líneas respetando lo que vemos en el último gráfico?
# Agrupamos el gráfico de líneas por especie! Ahora solo conecta por una línea las predicciones de una misma especie
(
    so.Plot(data = datos, x = "bill_length_mm")
    .add(so.Dot(), y = "bill_depth_mm")
    .add(so.Line(), y = "predicciones", color = "species") # Magia!
)

# No necesitamos agregar 3 líneas distintas, una para cada especie!

In [ ]:
# Y si ponemos el color en el Plot?
(
    so.Plot(data = datos, x = "bill_length_mm", color = "species")
    .add(so.Dot(), y = "bill_depth_mm")
    .add(so.Line(), y = "predicciones") # Magia!
)

En el modelo anterior, utilizamos la misma pendiente para todas las rectas y distintos intecepts. ¿Por qué? ¿Cuáles son los intercepts para cada especie? ¿Cómo podemos elegir las columnas para que los intercepts sean más claros?

In [ ]:
y, X = Formula("bill_depth_mm ~ bill_length_mm + species - 1").get_model_matrix(penguins)
X.head()

In [ ]:
# Ajustamos el modelo 
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo.fit(X, y)   # Realiza
print("Coeficientes:", modelo.coef_)

# Predecimos
y_pred = modelo.predict(X)
datos["predicciones"] = y_pred

# Graficamos
(
    so.Plot(data = datos, x = "bill_length_mm", color = "species")
    .add(so.Dot(), y = "bill_depth_mm")
    .add(so.Line(), y = "predicciones") # Milagro!
)

Obtenemos tres rectas paralelas (igual que antes). 

¿Cómo podemos modelar con 3 rectas distintas, una para cada especie?

In [ ]:
# Agregamos las interacciones!
y, X = Formula("bill_depth_mm ~ bill_length_mm * species").get_model_matrix(penguins)
X.head()

Tenemos 6 variables. Qué representa cada una?

In [ ]:
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo.fit(X, y)   # Realiza
print("Coeficientes:", modelo.coef_)

¿Cuál es la ecuación que modela bill_depth para la especie Chinstrap? ¿Y para Adelie?

In [ ]:
# Cómo hacemos para tener los coeficientes separados para cada especie?

# Qué esperamos con esta fórmula?
y, X = Formula("bill_depth_mm ~ bill_length_mm * species - 1").get_model_matrix(penguins)
X.head()

In [ ]:
# Y ahora?
y, X = Formula("bill_depth_mm ~ bill_length_mm * species - 1 - bill_length_mm").get_model_matrix(penguins)
X.head()

In [ ]:
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo.fit(X, y)   # Realiza
print("Coeficientes:", modelo.coef_)

**Pregunta:** ¿Cuáles coeficientes nos indican las pendientes de las rectas?

In [ ]:
# Calculamos el R^2
y_pred = modelo.predict(X)

# Calculando el R^2
r2 = r2_score(y, y_pred)
print('R^2: ', r2)

In [ ]:
# Los últimos tres números son las pendientes de las rectas.
# Cómo podemos graficar este modelo?

y_pred = modelo.predict(X)
datos = penguins.copy()

y_pred = modelo.predict(X)
datos["predicciones"] = y_pred
datos.head()


In [ ]:
(
    so.Plot(data = datos, x = "bill_length_mm", color = "species")
    .add(so.Dot(), y = "bill_depth_mm")
    .add(so.Line(), y = "predicciones")  # Magia!!
)

**Pregunta:** Considera que el sexo puede ser también una variable confusora? Que visualización podemos hacer para responder esta pregunta?

## Repaso

Según la AI de Google:

Los **machos de jirafa** son, en promedio, más altos que las **hembras**.  
Los machos suelen alcanzar una altura de **5 metros**, mientras que las hembras suelen medir alrededor de **4 metros**.  
Se han registrado ejemplares de jirafa machos con una altura de hasta **6 metros**.

**En detalle:**

- **Machos:** Miden entre **4,8 y 5,5 metros** de altura, con algunos ejemplares que pueden superar los 5 metros.  
- **Hembras:** Alcanzan una altura de **4,3 a 4,8 metros**.  
- **Diferencia:** La diferencia en altura entre machos y hembras se debe a factores como el tamaño y el desarrollo óseo, que se ven afectados por la **edad** y el **sexo**.

Si queremos ajustar la altura de las jirafas utilizando la edad y el sexo, ¿qué fórmula plantearíamos?

## Ejercicio

Volviendo al ejemplo de los exámenes SAT, realizar un modelo considerando la interacciones con la variable confusora que permita evaluar mejor la relación entre salario promedio y desempeño en el SAT.

**Sugerencia:** segmentar la variable confusora en dos o tres grupos.

In [ ]:
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/mosaicData/SAT.csv"
sat = pd.read_csv(url, index_col=0)
sat